# Thử nghiệm Đối chứng với Thư viện Scikit-Surprise

Notebook này sử dụng thư viện `scikit-surprise` để chạy các thuật toán tương tự trên cùng một tập dữ liệu MovieLens 100k.
Kết quả MAE và RMSE ở đây sẽ được dùng để đối chiếu trực tiếp với kết quả chạy từ thư mục `src/` (tự code từ đầu).

Các thuật toán cần kiểm chứng:
1. User-Based CF (Basic - Pearson)
2. User-Based CF (Means - Pearson)
3. Item-Based CF (Basic - Cosine)
4. Item-Based CF (Weighted/Adjusted Cosine - trong surprise sử dụng Pearson để gần tương đương)
5. SVD (Matrix Factorization)
\n

In [ ]:
import os
import pandas as pd
from surprise import Dataset, Reader, KNNBasic, KNNWithMeans, SVD, accuracy
from surprise.model_selection import train_test_split

# Đọc dữ liệu
data_path = '../data/raw/u.data'
reader = Reader(line_format='user item rating timestamp', sep='\t')
data = Dataset.load_from_file(data_path, reader=reader)

# Chia tập Train/Test (80/20)
# Lưu ý: Seed 42 giúp kết quả chia tập giống với custom code
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print("=== ĐÁNH GIÁ THƯ VIỆN SURPRISE ===")

# 1. User-Based Basic (Pearson)
print("\n1. User-Based Basic (Pearson)")
algo_u_basic = KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': True}, verbose=False)
algo_u_basic.fit(trainset)
preds_u_basic = algo_u_basic.test(testset)
accuracy.rmse(preds_u_basic)
accuracy.mae(preds_u_basic)

# 2. User-Based Means (Pearson)
print("\n2. User-Based Means (Pearson)")
algo_u_means = KNNWithMeans(k=40, sim_options={'name': 'pearson', 'user_based': True}, verbose=False)
algo_u_means.fit(trainset)
preds_u_means = algo_u_means.test(testset)
accuracy.rmse(preds_u_means)
accuracy.mae(preds_u_means)

# 3. Item-Based Basic (Cosine)
print("\n3. Item-Based Basic (Cosine)")
algo_i_basic = KNNBasic(k=40, sim_options={'name': 'cosine', 'user_based': False}, verbose=False)
algo_i_basic.fit(trainset)
preds_i_basic = algo_i_basic.test(testset)
accuracy.rmse(preds_i_basic)
accuracy.mae(preds_i_basic)

# 4. Item-Based (Pearson - Surprise không có Adjusted Cosine, đây là phương pháp gần giống nhất)
print("\n4. Item-Based (Pearson - Thay thế cho Adjusted Cosine)")
algo_i_pearson = KNNBasic(k=40, sim_options={'name': 'pearson', 'user_based': False}, verbose=False)
algo_i_pearson.fit(trainset)
preds_i_pearson = algo_i_pearson.test(testset)
accuracy.rmse(preds_i_pearson)
accuracy.mae(preds_i_pearson)

# 5. SVD
print("\n5. SVD (Funk SVD)")
algo_svd = SVD(n_factors=20, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
algo_svd.fit(trainset)
preds_svd = algo_svd.test(testset)
accuracy.rmse(preds_svd)
accuracy.mae(preds_svd)
\n

In [ ]:
print("\n======================================================\n")
print("=== ĐỐI CHỨNG VỚI MÔ HÌNH TỰ XÂY DỰNG TỪ ĐẦU (SCRATCH) ===\n")
print("======================================================\n")
import sys
sys.path.append("../")
from src.data_loader import load_matrix
from src.evaluation import compute_mae, compute_rmse
import pickle

test_matrix = load_matrix("../data/processed/test_matrix.npy")

with open("../models/user_cf.pkl", "rb") as f:
    user_cf = pickle.load(f)
with open("../models/item_cf.pkl", "rb") as f:
    item_cf = pickle.load(f)
    
from src.recommender import MatrixFactorizationSVD, UserBasedCollaborativeFiltering, ItemBasedCollaborativeFiltering
svd_model = MatrixFactorizationSVD()
svd_model.load_model("../models/svd_weights.pkl")

# 1. User-Based Basic (Pearson)
u_basic = UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='basic')
u_basic.train_matrix = user_cf.train_matrix
u_basic.similarity_matrix = user_cf.pearson_similarity_matrix
u_basic.user_means = user_cf.user_means

# 2. User-Based Means (Pearson)
u_means = UserBasedCollaborativeFiltering(k_neighbors=user_cf.k_neighbors, prediction_mode='means')
u_means.train_matrix = user_cf.train_matrix
u_means.similarity_matrix = user_cf.pearson_similarity_matrix
u_means.user_means = user_cf.user_means

# 3. Item-Based Basic (Cosine)
i_basic = ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors)
i_basic.train_matrix = item_cf.train_matrix
i_basic.similarity_matrix = item_cf.cosine_similarity_matrix

# 4. Item-Based Trọng số (Adjusted Cosine)
i_adj_cos = ItemBasedCollaborativeFiltering(k_neighbors=item_cf.k_neighbors)
i_adj_cos.train_matrix = item_cf.train_matrix
i_adj_cos.similarity_matrix = item_cf.adjusted_cosine_similarity_matrix

print("\n1. Custom User-Based Basic (Pearson)")
print(f"RMSE: {compute_rmse(test_matrix, u_basic):.4f}")
print(f"MAE:  {compute_mae(test_matrix, u_basic):.4f}")

print("\n2. Custom User-Based Means (Pearson)")
print(f"RMSE: {compute_rmse(test_matrix, u_means):.4f}")
print(f"MAE:  {compute_mae(test_matrix, u_means):.4f}")

print("\n3. Custom Item-Based Basic (Cosine)")
print(f"RMSE: {compute_rmse(test_matrix, i_basic):.4f}")
print(f"MAE:  {compute_mae(test_matrix, i_basic):.4f}")

print("\n4. Custom Item-Based Trọng số (Adjusted Cosine)")
print(f"RMSE: {compute_rmse(test_matrix, i_adj_cos):.4f}")
print(f"MAE:  {compute_mae(test_matrix, i_adj_cos):.4f}")

print("\n5. Custom SVD")
print(f"RMSE: {compute_rmse(test_matrix, svd_model):.4f}")
print(f"MAE:  {compute_mae(test_matrix, svd_model):.4f}")
\n